# Transformer imputer — training & constraint ablation (8 runs)

Bidirectional **iTransformer** imputer, fixed architecture (SAITS-style value⊕mask
embedding, channels-as-tokens, T = 288 = one day, joint masking of all 6 dispatch
channels). Max 100 epochs, early stop patience 10 on validation gap-MAE.

**Stage 1 — training-method ablation (constraint = none), 4 runs.** Pick **T\*** =
lowest validation *blackout*-MAE.

| ID | gap sampling | flank aug | aux recon loss |
| --- | --- | --- | --- |
| T1 | mixed: \|G\| ∈ {3h, 6h, 12h}, +20% blackout | off | on (0.1) |
| T2 | mixed | on: flanks ×(1+δ), δ~U(−0.1,0.1), p=0.5 | on |
| T3 | blackout only (no flanks) | n/a | on |
| T4 | mixed | on | **off** |

**Stage 2 — constraint-method ablation (training = T\*), 4 runs.**

| ID | mechanism | guarantee | runs |
| --- | --- | --- | --- |
| C0 | none (= T\*) | no | 0 |
| C1 | soft penalty, λ ∈ {0.1, 1, 10} on val | no | 3 |
| C2 | post-hoc projection Π of C0 (inference only) | **yes** | 0 |
| C3 | RAYEN fixed-D over the stacked gap output | **yes** | 1 |

**Deliverables:** D1 training + constraint tables (single numbers, causal + pro-rata
reference rows, feasibility floor F footnote) · D2 4-day stacked figure (actual /
causal closed-loop / best constrained, blackout mode) · D3 counterfactual change
report at **q\* = 3.699%** — the corrected energy-conserving `FixedPercentageShift`
(uniform off-window reduction shifted into 11:00–14:00, price→0, renewables fixed),
whole-day blackout, **Option A: no dispatch pinned anywhere**. q\* is the largest
equal shift dispatch-feasible on ALL 186 test days (`shift_feasibility.py`; the
demand-cap q_max = 4.856% is infeasible on 63 days).

**Tie rule (no seeds):** margins < ~5% relative MAE between arms are TIES, not wins —
only clear gaps drive T\* and the C-winner. If a tie matters (e.g. C2 vs C3 for the
paper's claim), rerun just that pair with 2–3 seeds.

> Scripts live in `imputation/itr_*.py` — they must be **committed & pushed** before
> this notebook can use them in Colab.

In [ ]:
REPO = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""   # PRIVATE repo: fine-grained READ-ONLY token (Contents: read)
import os
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
# ABSOLUTE paths so re-running can never nest clones or change cwd
ROOT = "/content/energy_modelling"
if not os.path.exists(ROOT):
    !git clone -q $url $ROOT
%cd $ROOT
!git pull -q
!nvidia-smi -L

## 1 · Sanity — data windows + the q\* feasibility certificate

`itr_data.py` self-test (window sampler, mixed/blackout, full test days) and
`shift_feasibility.py` (certifies q\* = 3.699% feasible on all days under Option-A
whole-day constraints — the counterfactual's foundation).

In [ ]:
!python imputation/itr_data.py
!python imputation/shift_feasibility.py

## 2 · Stage 1 — train T1–T4 (GPU)

Each run early-stops on its own validation gap-MAE and records the common
validation blackout-MAE (the T\* selection metric) in its json.

In [ ]:
!python imputation/itr_train.py --arm T1

In [ ]:
!python imputation/itr_train.py --arm T2

In [ ]:
!python imputation/itr_train.py --arm T3

In [ ]:
!python imputation/itr_train.py --arm T4

In [ ]:
# D1 training table + T* selection (5% tie rule applied automatically)
!python imputation/itr_bench.py --stage1
from IPython.display import Markdown, display
display(Markdown(open("imputation/results/itr/d1_training_table.md").read()))

## 3 · Stage 2 — constraint arms on T\* (4 runs)

C1 = 3 soft-penalty runs (λ picked on val); C3 = 1 RAYEN run (trains *inside* the
differentiable ray-shoot). C0 and C2 need no training — C2 is the inference-only
projection of C0.

In [ ]:
import json
TSTAR = json.load(open("imputation/results/itr/stage1_selection.json"))["tstar"]
print("T* =", TSTAR)
!python imputation/itr_train.py --arm $TSTAR --constraint soft --lam 0.1
!python imputation/itr_train.py --arm $TSTAR --constraint soft --lam 1
!python imputation/itr_train.py --arm $TSTAR --constraint soft --lam 10

In [ ]:
import json
TSTAR = json.load(open("imputation/results/itr/stage1_selection.json"))["tstar"]
!python imputation/itr_train.py --arm $TSTAR --constraint rayen

In [ ]:
# D1 constraint table: C0-C3 + causal & pro-rata references + feasibility floor F
!python imputation/itr_bench.py --stage2
from IPython.display import Markdown, display
display(Markdown(open("imputation/results/itr/d1_constraint_table.md").read()))

## 4 · D2 — 4-day stacked figure (actual / causal closed-loop / best constrained)

Blackout mode. The causal panel feeds each 12h chunk's fill forward as the next
chunk's context — drift and the reconnection seam are the point. The constrained
panel is per-day blackout imputation through the stage-2 winner's guarantee map.

In [ ]:
!python imputation/itr_figures.py --which d2
from IPython.display import Image, display
display(Image("imputation/results/itr/figure/d2_stack_4day.png"))

## 5 · D3 — counterfactual change report at q\* (Option A, nothing pinned)

Whole-day blackout, base vs shifted drivers, free-endpoint guarantee map on both
runs. Read: ΔEᵢ per fuel free/off-window, peaks, **violations must be 0**, and the
conservation check Σᵢ SIGNᵢ·ΔEᵢ ≈ shifted net-demand energy in each region.

In [ ]:
!python imputation/itr_figures.py --which d3
from IPython.display import Image, Markdown, display
display(Markdown(open("imputation/results/itr/d3_report.md").read()))
display(Image("imputation/results/itr/figure/d3_counterfactual.png"))

In [ ]:
# EXPORT the checkpoints before the runtime dies (the bi-LSTM lesson: only the
# numbers survived last time). Downloads every itr ckpt + json + the tables.
from google.colab import files
import glob
for p in sorted(glob.glob("imputation/results/itr/itr_*.pt") +
                glob.glob("imputation/results/itr/*.json") +
                glob.glob("imputation/results/itr/*.md")):
    files.download(p)

## 6 · Conclusions — fill in after the run

- **T\*** = ____ (val blackout-MAE ____; ties within 5%: ____)
- **Best guaranteed-feasible** = ____ (C2 vs C3 blackout MAE ____ vs ____; tie? ____)
- Does any constrained learner beat the pro-rata and causal references? ____
- Feasibility floor: F_Π = ____, F_rayen = ____ (if F_rayen ≈ C3's MAE, the RAYEN
  single-α conservatism — not the network — is the binding cost)
- D3 at q\* = 3.699%: violations ____ (must be 0); conservation gaps ____ MWh
- Remember the tie rule: <5% relative MAE = tie; rerun only a deciding pair with seeds.